# Avance 2 – Clasificador de intenciones (procesamiento de lenguaje natural)
**Proyecto:** Asistente de reclutamiento del restaurante El Sauce
**Alumna:** Juana Paola Puente Huerta

En el Avance 1 el chatbot identificaba las intenciones buscando **palabras clave exactas**, por eso no entendía
mensajes como «me intersa» o «cuanto pagan d mesra». En este cuaderno se entrena un **clasificador de intenciones**
que aprende a partir de ejemplos:

1. Conjunto de datos textual (frases etiquetadas con su intención)
2. Preprocesamiento: tokenización, normalización, eliminación de *stopwords* y *stemming*
3. Representación con **TF-IDF**
4. Clasificación con **Naive Bayes** y **SVM**, y comparación de su desempeño

## 1. Instalación y configuración

In [ ]:
import re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

nltk.download('stopwords', quiet=True)
SEMILLA = 42

## 2. Conjunto de datos textual
El archivo `dataset_intenciones.csv` contiene frases escritas como las mandaría un candidato por WhatsApp:
con y sin acentos, con faltas de ortografía, abreviaciones y distintas formas de preguntar lo mismo.
Cada frase está etiquetada con una de **15 intenciones**. Respecto al Avance 1 se agregó la intención
`consulta_rh` para preguntas laborales que no están en la base de conocimiento (seguro social, aguinaldo, contrato),
y `fuera_de_tema` para mensajes que no tienen que ver con el reclutamiento.

> Sube `dataset_intenciones.csv` al panel de archivos de Colab (ícono de carpeta) antes de ejecutar.

In [ ]:
datos = pd.read_csv('dataset_intenciones.csv')
print("Total de frases:", len(datos), "| Intenciones:", datos['intencion'].nunique())
datos.sample(8, random_state=SEMILLA)

In [ ]:
conteo = datos['intencion'].value_counts().sort_values()
conteo.plot(kind='barh', figsize=(7, 5), color='#2E6B4F')
plt.title('Frases por intención'); plt.xlabel('Número de frases')
plt.tight_layout(); plt.show()

## 3. Preprocesamiento del texto
| Paso | Qué hace | Ejemplo |
|---|---|---|
| **Normalización** | Minúsculas, quita acentos y signos | «¿Cuánto PAGAN?» → «cuanto pagan» |
| **Tokenización** | Separa el texto en palabras con NLTK | «cuanto pagan de mesera» → [cuanto, pagan, de, mesera] |
| **Stopwords** | Quita palabras muy comunes sin significado propio | [cuanto, pagan, ~~de~~, mesera] |
| **Stemming** | Reduce cada palabra a su raíz con el *SnowballStemmer* | pagan, paga, pago → pag |

De la lista de *stopwords* de NLTK se **conservan** palabras que sí ayudan a distinguir intenciones,
como «cuanto», «donde», «cuando», «como», «sin» y «no» (por ejemplo, «**sin** experiencia» o «**donde** están»).

In [ ]:
stemmer = SnowballStemmer('spanish')

def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')   # quita acentos
    texto = re.sub(r'[^a-z0-9 ]', ' ', texto)                               # quita signos
    return re.sub(r'\s+', ' ', texto).strip()

CONSERVAR = {'cuanto', 'cuanta', 'cuantos', 'cuantas', 'donde', 'cuando', 'como', 'cual', 'cuales', 'sin', 'no'}
STOPWORDS = {normalizar(p) for p in stopwords.words('spanish')} - CONSERVAR

def preprocesar(texto, detalle=False):
    paso1 = normalizar(texto)
    paso2 = wordpunct_tokenize(paso1)
    paso3 = [t for t in paso2 if t not in STOPWORDS]
    paso4 = [stemmer.stem(t) for t in paso3]
    if detalle:
        return {'original': texto, 'normalizado': paso1, 'tokens': paso2, 'sin_stopwords': paso3, 'stemming': paso4}
    return ' '.join(paso4) if paso4 else paso1   # si todo era stopword, se deja el texto normalizado

print("Stopwords de NLTK usadas:", len(STOPWORDS))

In [ ]:
# Ejemplo paso a paso
ejemplos = ['¿Cuánto PAGAN de mesera?', 'Puedo trabajar sin experiencia?', 'Me intersa el puesto de cocinero']
pd.DataFrame([preprocesar(e, detalle=True) for e in ejemplos])

In [ ]:
datos['procesado'] = datos['texto'].apply(preprocesar)
datos[['texto', 'procesado', 'intencion']].head(10)

## 4. Representación con TF-IDF y clasificadores
**TF-IDF** convierte cada frase en un vector de números: le da más peso a las palabras que aparecen en una frase
pero son poco comunes en el resto (por ejemplo «propin» o «sueld»), y menos a las que aparecen en todas.

Se comparan tres modelos:
- **A. Naive Bayes** con TF-IDF de palabras.
- **B. SVM lineal** con TF-IDF de palabras.
- **C. SVM lineal** con TF-IDF de palabras **+ fragmentos de 2 a 4 letras** (n-gramas de caracteres).
  Los fragmentos permiten reconocer palabras mal escritas: «intersa» comparte casi todos sus fragmentos con «interesa».

In [ ]:
def tfidf_palabras():
    return TfidfVectorizer(ngram_range=(1, 2))

def tfidf_palabras_y_letras():
    return FeatureUnion([
        ('palabras', TfidfVectorizer(ngram_range=(1, 2))),
        ('letras',   TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))),
    ])

modelos = {
    'A. Naive Bayes (palabras)':        Pipeline([('tfidf', tfidf_palabras()), ('clf', MultinomialNB(alpha=0.3))]),
    'B. SVM (palabras)':                Pipeline([('tfidf', tfidf_palabras()), ('clf', LinearSVC(C=1.0))]),
    'C. SVM (palabras + letras)':       Pipeline([('tfidf', tfidf_palabras_y_letras()), ('clf', LinearSVC(C=1.0))]),
}

## 5. Entrenamiento y evaluación
Se separa el **20 %** de las frases como conjunto de prueba (estratificado por intención).
Además se usa **validación cruzada de 5 particiones** sobre todo el conjunto, que da una medida más estable
porque cada frase se usa como prueba una vez.

In [ ]:
X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
    datos['procesado'], datos['intencion'], datos['texto'],
    test_size=0.2, stratify=datos['intencion'], random_state=SEMILLA)
print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    acc_test = accuracy_score(y_test, modelo.predict(X_test))
    cv_scores = cross_val_score(modelo, datos['procesado'], datos['intencion'], cv=cv, scoring='accuracy')
    resultados.append({'Modelo': nombre, 'Accuracy prueba': round(acc_test, 3),
                       'Accuracy val. cruzada': f"{cv_scores.mean():.3f} ± {cv_scores.std():.3f}"})
pd.DataFrame(resultados)

### Reporte detallado del mejor modelo (C)

In [ ]:
mejor = modelos['C. SVM (palabras + letras)']
y_pred = mejor.predict(X_test)
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

In [ ]:
etiquetas = sorted(datos['intencion'].unique())
cm = confusion_matrix(y_test, y_pred, labels=etiquetas)
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(cm, display_labels=etiquetas).plot(ax=ax, cmap='Greens', colorbar=False, xticks_rotation=60)
ax.set_title('Matriz de confusión – clasificador de intenciones (prueba)')
plt.tight_layout(); plt.show()

In [ ]:
# Frases de prueba mal clasificadas
errores = pd.DataFrame({'frase': t_test, 'real': y_test, 'predicha': y_pred})
errores[errores['real'] != errores['predicha']]

## 6. Confianza de la predicción
Un SVM siempre elige una intención, aunque el mensaje no tenga sentido. Para detectar cuando **no está seguro**,
se calibra el modelo para obtener probabilidades. Si la probabilidad más alta es menor a un umbral,
el chatbot responderá que no entendió y mostrará los temas disponibles.

In [ ]:
clasificador = Pipeline([
    ('tfidf', tfidf_palabras_y_letras()),
    ('clf', CalibratedClassifierCV(LinearSVC(C=1.0), cv=3)),
])
clasificador.fit(datos['procesado'], datos['intencion'])   # modelo final: con todas las frases
UMBRAL_TEXTO = 0.30

def clasificar_intencion(mensaje, umbral=UMBRAL_TEXTO):
    p = clasificador.predict_proba([preprocesar(mensaje)])[0]
    i = p.argmax()
    intencion = clasificador.classes_[i] if p[i] >= umbral else 'no_entendido'
    return intencion, round(float(p[i]), 2)

### Prueba con frases nuevas
Ninguna de estas frases está en el conjunto de datos. Se incluyen los casos que fallaron en el Avance 1.

In [ ]:
frases_nuevas = [
    'me intereza mucho el trabajo',
    'cuanto pagan d mesra',
    'q horario manejan',
    'ocupan chef?',
    'buenas tardes, informacion de la vacante porfa',
    'dan seguro social?',
    'tengo 0 experiencia, me contratan?',
    'en donde se ubican',
    'cuanto sale el pozole',
    'xyz qwerty',
]
pd.DataFrame([(f, *clasificar_intencion(f)) for f in frases_nuevas],
             columns=['Mensaje', 'Intención detectada', 'Confianza'])